In [ ]:
!pip install mlflow boto3 awscli optuna xgboost imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/1

In [ ]:
from google.colab import userdata
import os


os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

In [ ]:

import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/")

In [ ]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos for yt")

2026/07/19 18:17:11 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 - ML Algos for yt' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/213dc38cd2f74c94aea47e2200a269f4', creation_time=1784485031768, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1784485031768, lifecycle_stage='active', name='Exp 5 - ML Algos for yt', tags={}, trace_location=None, workspace='default'>

In [ ]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


In [ ]:
df = pd.read_csv("/content/yt_preprocessed_data.csv").dropna(subset=['Comment'])
df.shape

(17680, 3)

In [ ]:
df.drop(columns=[ "Unnamed: 0"], inplace=True)

In [ ]:
print(df["Sentiment"].unique())
print(df["Sentiment"].value_counts(dropna=False))
print(df["Sentiment"].dtype)

['neutral' 'negative' 'positive']
Sentiment
positive    11035
neutral      4329
negative     2316
Name: count, dtype: int64
object


In [ ]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df["Sentiment"] = df["Sentiment"].map({
    "neutral": 0,
    "positive": 1,
    "negative": 2
})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

ngram_range = (1, 3)  # Trigram setting
max_features = 1000  # Set max_features to 1000 for Bow

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['Comment'], df['Sentiment'], test_size=0.2, random_state=42, stratify=df['Sentiment'])

vectorizer = CountVectorizer(
    ngram_range=(1, 3),
    max_features=1000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Apply Random Oversampling
ros = RandomOverSampler(random_state=42)

X_train_vec, y_train = ros.fit_resample(
    X_train_vec,
    y_train
)


# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTETomek_TFIDF_bigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path=f"{model_name}_model",
            serialization_format="pickle"
        )


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))


# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)

# Run the experiment for XGBoost
run_optuna_experiment()


[I 2026-07-19 18:17:27,517] A new study created in memory with name: no-name-77b09d51-19a3-4c37-920a-c913600e0a48
[I 2026-07-19 18:17:40,111] Trial 0 finished with value: 0.7121040723981901 and parameters: {'n_estimators': 289, 'learning_rate': 0.06698998980181432, 'max_depth': 10}. Best is trial 0 with value: 0.7121040723981901.
[I 2026-07-19 18:17:51,484] Trial 1 finished with value: 0.5811651583710408 and parameters: {'n_estimators': 163, 'learning_rate': 0.0031425735122687748, 'max_depth': 10}. Best is trial 0 with value: 0.7121040723981901.
[I 2026-07-19 18:18:05,289] Trial 2 finished with value: 0.5667420814479638 and parameters: {'n_estimators': 207, 'learning_rate': 0.0002300567488623265, 'max_depth': 10}. Best is trial 0 with value: 0.7121040723981901.
[I 2026-07-19 18:18:20,009] Trial 3 finished with value: 0.5805995475113123 and parameters: {'n_estimators': 219, 'learning_rate': 0.0024453496953519773, 'max_depth': 10}. Best is trial 0 with value: 0.7121040723981901.
[I 2026-

🏃 View run XGBoost_SMOTETomek_TFIDF_bigrams at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/13/runs/8238167a71754850bc422783cc2e1537
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/13


In [ ]:
import pandas as pd
from mlflow.tracking import MlflowClient

# Replace with your Experiment ID
EXPERIMENT_ID = "YOUR_EXPERIMENT_ID"

client = MlflowClient()

# Get all runs in the experiment
runs = client.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    max_results=100000
)

print(f"Total Runs: {len(runs)}")

rows = []

for run in runs:
    row = {
        "run_id": run.info.run_id,
        "run_name": run.data.tags.get("mlflow.runName", "")
    }

    # Copy all metrics
    row.update(run.data.metrics)

    # Copy all parameters
    row.update(run.data.params)

    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv("experiment_runs.csv", index=False)

print(df.head())
print("Saved to experiment_runs.csv")